# 01 — Collecter PokéAPI par CSV et par API

**Objectif :** obtenir deux tables au même format, une ligne par forme, puis vérifier qu’elles contiennent les mêmes observations et les mêmes valeurs avant une future fusion avec Showdown. Aucune forme n’est filtrée ou regroupée, même si ses statistiques sont identiques à celles d’une autre forme. Talents et conditions d’évolution sont hors périmètre.

Les fonctions sont dans `src/collect.py`, les adresses et instantanés dans `config/sources.json`. Exécuter les cinq cellules de code dans l’ordre pour vérifier les deux méthodes. La collecte CSV et la collecte API sont deux cellules séparées : après l’import, on peut aussi lancer uniquement celle dont on a besoin.

**Deux versions différentes peuvent diverger :** les CSV sont figés à une révision du dépôt, tandis que l’API publique est actuelle au moment du téléchargement. Le code vérifie l’égalité ; il ne modifie pas une source pour la forcer.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebook":
    project_dir = project_dir.parent
assert (project_dir / "src" / "collect.py").exists(), "Ouvrir depuis le projet ou notebook/."
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.collect import (
    collect_sources, load_sources, check_sources, pokeapi_from_csv,
    collect_pokeapi_api, compare_pokeapi, export_collections,
)

## 1. Méthode CSV : télécharger, contrôler, mettre au format commun

Les fichiers bruts restent séparés et inchangés. `collect_sources()` vérifie leurs empreintes et réutilise l’instantané présent. Cette étape conserve aussi Showdown, qui sera utilisé plus tard : **il n’entre pas dans la table PokéAPI comparée**.

`pokeapi_from_csv()` rattache à chaque forme sa variété, son espèce, ses six statistiques, son groupe de jeux et ses noms. Les types explicites de forme ont priorité ; à défaut, on utilise ceux de la variété. Il s’agit d’une mise au même format que l’API, pas d’une correction des sources.

In [ ]:
data_dir = collect_sources()
data = load_sources()
source_checks = check_sources(data)
pokeapi_csv = pokeapi_from_csv(data)
print("Table CSV :", pokeapi_csv.shape)
display(pokeapi_csv.head())

## 2. Méthode API : parcourir toutes les formes et leurs références

On parcourt **toutes les pages du catalogue `/pokemon-form/`**, indépendamment de la liste des CSV. Chaque forme fournit l’adresse de sa variété et de son groupe de jeux ; la variété fournit l’adresse de son espèce et les statistiques. Ainsi, aucune génération n’est fixée à la main et la génération de l’espèce ne remplace pas celle de la forme.

Les JSON sont sauvegardés dans `data/raw/api_01/` avec adresse, date et empreinte. La première collecte peut durer plusieurs minutes ; les suivantes réutilisent le cache. Une erreur HTTP arrête la collecte, sans repli silencieux. Après une interruption, relancer cette cellule reprend les téléchargements manquants. L’API peut évoluer durant une collecte : les dates de chaque réponse sont conservées.

Les types viennent de la forme API, y compris pour Arceus et Silvallié. Les statistiques viennent directement de sa variété : aucun appel `/pokemon/nom-de-forme` n’est tenté.

In [ ]:
pokeapi_api = collect_pokeapi_api()
print("Table API :", pokeapi_api.shape)
display(pokeapi_api.head())

# 02 — Comprendre les sources avant de préparer la base

**Question du projet :** les générations récentes introduisent-elles des Pokémon aux statistiques plus élevées ou plus spécialisées ? Avant de comparer les générations, nous devons comprendre les observations et les informations disponibles.

**Choix retenu : une ligne par forme PokéAPI.** Nous conservons toutes les formes de l’instantané, même cosmétiques, Gigamax ou non définies par défaut. Garder le détail permettra ensuite des analyses par forme, variété, espèce ou génération. Les talents et les conditions d’évolution sont hors périmètre.

Les fonctions sont dans `src/exploration.py`. Les cellules suivent l’ordre **observation → explication → conclusion ou proposition de traitement**. Aucun fichier brut n’est modifié, aucun regroupement de formes n’est appliqué et aucune base finale n’est exportée. Les résultats chiffrés commentés concernent `reference_01`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebook":
    project_dir = project_dir.parent
assert (project_dir / "src" / "exploration.py").exists(), "Ouvrir depuis le projet ou notebook/."
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.collect import collect_sources, load_sources, check_sources
from src import exploration as eda

collect_sources()  # Vérifie et réutilise l’instantané partagé.
data = load_sources()
source_checks = check_sources(data)

## 1. Que représente une ligne ?

Une **espèce** peut avoir plusieurs **variétés**, et une variété plusieurs **formes**. Les six statistiques sont stockées au niveau de la variété. Nous rattachons ces informations aux formes par des jointures qui contrôlent le nombre de correspondances. La fonction vérifie que tous les `form_id` restent présents et uniques.

Les exemples ci-dessous sont choisis pour expliquer la structure ; le traitement lui-même s’applique à toutes les formes.

In [ ]:
forms = eda.explore_forms(data)
display(eda.summarize_forms(forms))
species = eda.summarize_species(forms)
display(species.loc[species["species_id"].isin([493, 669, 745, 773, 925, 1007, 1008])])

**Constat :** 1 579 formes représentent 1 351 variétés et 1 025 espèces. Flabébé a cinq formes pour une variété ; Lougaroc a trois formes pour trois variétés. Arceus a 19 formes et Silvallié 18, chacun pour une seule variété.

**Conclusion :** les identifiants `form_id`, `pokemon_id` et `species_id` désignent des objets différents. Nous gardons les 236 formes non définies par défaut et les 34 Gigamax. Les huit variétés de déplacement de Koraidon/Miraidon n’ont pas de forme marquée par défaut, mais leurs formes sont bien présentes : aucun traitement de remplacement n’est nécessaire pour les conserver.

## 2. Statistiques identiques et dates différentes : des doublons ?

Les six statistiques viennent de la variété. La génération vient du groupe de jeux d’introduction de **chaque forme**. Nous gardons la génération de l’espèce séparément pour ne pas confondre les dates.

In [ ]:
examples = ["charizard", "charizard-mega-x", "charizard-gmax", "lycanroc-midday",
            "lycanroc-midnight", "lycanroc-dusk", "maushold-family-of-four", "maushold-family-of-three"]
display(forms.loc[forms["form_name"].isin(examples),
                  ["form_name", "form_generation"] + eda.STAT_COLUMNS])
display(forms.loc[forms["form_name"].isin(["arceus-normal", "arceus-fairy"]),
                  ["form_name", "pokemon_id", "introduction_games", "form_generation", "species_generation"]])

**Constat :** les profils de Lougaroc diffèrent ; ceux de Famignol sont identiques. Dracaufeu et Dracaufeu Gigamax ont les mêmes six statistiques, mais leurs formes sont introduites respectivement en générations 1 et 8. Arceus Fée apparaît après Arceus Normal tout en partageant sa variété.

**Conclusion :** ni les statistiques identiques ni une même variété ne justifient de supprimer une forme. Les statistiques et les générations sont renseignées pour les 1 579 formes. Aucune date de forme ne précède ici celle de son espèce ; ce contrôle ne constitue toutefois pas une validation historique indépendante de toutes les dates.

**Limite :** les statistiques sont celles de l’instantané, pas nécessairement celles qu’avait le Pokémon à son introduction. Une moyenne par forme donne aussi davantage de poids aux espèces ayant beaucoup de formes. Le niveau et la pondération devront être précisés pour chaque analyse.

## 3. Un type absent dans une table est-il inconnu ?

Les types des variétés et les types explicitement renseignés pour les formes restent séparés. La recherche suivante repère toutes les variétés concernées à partir des données, sans sélectionner les noms Arceus ou Silvallié dans la règle.

In [ ]:
form_types = eda.inspect_form_types(forms)
display(form_types)

**Constat :** 35 formes ont un type explicitement renseigné, et 1 544 n’en ont pas dans cette table partielle. Toutes disposent néanmoins de types enregistrés au niveau de leur variété. La recherche générale retrouve Arceus et Silvallié.

**Explication :** les formes Normal n’ont pas de type répété dans la table des formes ; les autres formes peuvent y avoir un type différent de celui de la variété. `???` pour `arceus-unknown` est un libellé explicite, pas un `NaN`. Cette entrée historique inutilisée est conservée et signalée : notre inventaire ne se limite pas aux formes normalement accessibles en jeu. [Description d’Arceus](https://bulbapedia.bulbagarden.net/wiki/Arceus).

**Proposition non appliquée :** utiliser les types explicites de la forme lorsqu’ils existent, sinon ceux de la variété, en gardant la provenance. Nous ne remplaçons aucune valeur à ce stade.

## 4. Comprendre les autres valeurs manquantes de PokéAPI

Le premier tableau recense les absences dans toutes les tables CSV collectées. Les autres cherchent leurs raisons. Une ligne du tableau des traductions est une espèce dans une langue, pas une forme supplémentaire.

In [ ]:
display(eda.missing_pokeapi(data))
missing_context = eda.missing_pokeapi_context(data)
display(missing_context["habitat"])
display(missing_context["varieties"])
display(missing_context["labels"])
display(missing_context["translations"])

| Constat dans l’instantané | Explication et proposition de traitement |
|---|---|
| 639 habitats absents | Toutes les espèces des générations IV à IX sont concernées, contre aucune des générations I à III. Ne pas utiliser ce champ pour comparer toutes les générations sans source complémentaire ; ne pas imputer un habitat majoritaire. |
| 49 expériences de base absentes | Elles concernent uniquement des variétés non définies par défaut. La cause exacte de chaque absence n’est pas établie. Garder les valeurs manquantes : ce champ n’est pas nécessaire aux six statistiques. |
| 139 valeurs `order` absentes | Il s’agit d’un ordre de présentation, pas de l’identifiant. Utiliser les identifiants pour les jointures. |
| 961 libellés courts de forme absents | Leurs identifiants complets sont tous disponibles ; toutes ces formes sont marquées par défaut. Afficher `identifier`, sans supprimer les formes ni inventer un suffixe `normal`. |
| Descriptions `genus` manquantes selon la langue | Elles sont distinctes des noms. Les lignes françaises (`local_language_id = 5`) sont complètes ; choisir explicitement une langue et ne pas inventer de traduction. |
| 541 liens `evolves_from_species_id` absents | Aucun prédécesseur n’est renseigné dans ce champ ; ce n’est pas une statistique manquante. Conserver l’absence, sans construire de catégorie évolutive dans notre périmètre. |
| 825 `conquest_order` absents | Cet ordre concerne Pokémon Conquest, pas l’ensemble du Pokédex. Ne pas l’utiliser comme identifiant ou ordre universel. |
| Métadonnées manquantes dans `stats` | Les PV et la vitesse n’ont pas de classe physique/spéciale ; précision et esquive n’ont pas d’index renseigné. Ce ne sont pas des absences parmi les six valeurs de base étudiées. |

**Conclusion :** aucune suppression générale des lignes incomplètes ne convient. Elle éliminerait notamment les espèces récentes sans habitat et de nombreuses formes dont seul le nom court manque. Nous conservons les observations et distinguons les absences non applicables, les informations disponibles ailleurs et les véritables lacunes. [Définition des champs PokéAPI](https://pokeapi.co/docs/v2).

## 5. Que manque-t-il dans Showdown, et que peut-il nous apporter ?

Showdown servira à **vérifier les informations communes après fusion** et à ajouter **Paradoxe, Ultra-Chimère et tier**. Les statuts légendaire/fabuleux sont déjà disponibles dans PokéAPI.

Nous distinguons un champ omis, une valeur explicitement nulle et un contenu vide. Ces nombres portent sur les 1 518 fiches Showdown ; ils ne sont pas encore des taux de couverture des 1 579 formes PokéAPI.

In [ ]:
showdown = data["showdown_pokedex"]
display(eda.missing_showdown(showdown))

**Constat :** 37 fiches omettent les champs communs tels que le numéro, les statistiques ou les types. Le tier manque dans 102 fiches, les tags dans 1 358 et la génération explicite dans 1 351. Les champs étudiés sont omis, pas remplis par `null` ou une liste vide. Ces absences n’ont pas toutes la même signification : certaines fiches renvoient à une autre fiche.

Nous examinons ces références sans recopier leur contenu dans les données. Les colonnes `tier_base` et `tier_combat` ci-dessous sont des valeurs candidates à examiner, pas des tiers attribués.

In [ ]:
references = eda.inspect_showdown_references(showdown)
display(references["summary"])
display(references["tier_conflicts"])

**Les 37 fiches cosmétiques** renvoient toutes à une fiche contenant les cinq champs communs. Il s’agit d’informations non répétées. Un héritage documenté sera envisageable pour ces fiches explicitement cosmétiques ; il ne doit pas être généralisé aux formes qui changent de type ou de statistiques.

**Les 102 tiers absents** ont tous une valeur dans la fiche `baseSpecies`, mais la copier systématiquement serait incorrect : pour deux transformations d’Ogerpon, cette référence porte `OU`, alors que la référence de combat porte `Uber` ou `UUBL`. Le [moteur Showdown](https://github.com/smogon/pokemon-showdown/blob/master/sim/dex-species.ts) prévoit des règles d’héritage et des cas particuliers. La règle applicable devra être documentée avant traitement ; le moteur courant n’est pas une version figée avec notre JSON. `Illegal` est une valeur de contexte, pas une absence ni un motif d’exclusion.

**Les tags absents ne signifient pas automatiquement « non Paradoxe » ou « non Ultra-Chimère ».** Parmi ces fiches, 94 renvoient à une fiche contenant des tags. Cela donne des pistes d’héritage, sans garantir l’exhaustivité des étiquettes de base. Les 16 fiches étiquetées `Paradox` et les 11 étiquetées `Ultra Beast` sont des comptages de mentions, pas une classification complète validée. Conserver les tags bruts et un statut indéterminé tant que la catégorie n’a pas été vérifiée.

**Limites :** le tier dépend du contexte compétitif et de la date, pas seulement des statistiques de base. La génération des formes est renseignée via PokéAPI. Les champs structurels `baseSpecies` ou `forme` ne sont pas nécessaires sur toutes les fiches de base. Aucun remplissage automatique n’est appliqué ici.

## 6. Les mêmes statistiques suffisent-elles pour fusionner ?

Nous cherchons les fiches Showdown ayant le même numéro d’espèce et les mêmes six valeurs. Cette recherche est seulement un diagnostic : elle ne valide ni l’identité de la forme, ni son type, ni sa génération. Les candidats restent séparés de notre inventaire.

In [ ]:
candidates = eda.inspect_showdown_candidates(forms, showdown)
display(candidates["summary"])
display(candidates["candidates"].loc[
    candidates["candidates"]["form_name"].isin(["charizard", "charizard-gmax"]),
    ["form_name", "showdown_name"]])

**Constat :** aucune forme n’est sans candidat numérique, 1 236 en ont un et 343 en ont plusieurs. Dracaufeu et Dracaufeu Gigamax illustrent pourquoi l’égalité des statistiques ne suffit pas à reconnaître une forme. Même un candidat unique n’est pas une identité validée ; une forme historique inutilisée peut partager un profil avec des fiches jouables.

**Conclusion pour la préparation :** établir une correspondance de forme documentée, conserver toutes les formes PokéAPI par une jointure à gauche, puis comparer séparément les valeurs des deux sources. Les formes non rapprochées ou ambiguës restent présentes. Une vérification des statistiques ne sera pas indépendante si l’on a choisi la correspondance uniquement pour obtenir des statistiques égales. L’accord entre deux sources peut aussi refléter une origine commune de leurs informations.

## Bilan : ce que nous retenons pour la suite

- **Population :** 1 579 formes, avec leurs identifiants de variété et d’espèce ; aucun regroupement, filtre par défaut ou retrait des Gigamax.
- **Informations disponibles :** les six statistiques et la génération sont renseignées pour chaque forme. La provenance des types reste à expliciter lors de la préparation.
- **Valeurs manquantes :** conserver les absences non utiles/non applicables ; examiner les héritages de fiches explicitement documentés ; ne pas déduire une catégorie négative d’un tag absent. Aucun traitement proposé n’est encore appliqué.
- **Fusion Showdown :** vérifier l’identité des correspondances avant d’ajouter les catégories et le tier. Mesurer ensuite la couverture sur les formes, en distinguant absence de correspondance et champ absent dans une fiche identifiée.
- **Limites :** exhaustivité relative aux sources, entrées historiques inutilisées, statistiques actuelles plutôt qu’historiques, répétition des profils entre formes et poids des espèces riches en formes. Les talents et conditions d’évolution sont hors périmètre.

Le choix de conserver le détail permet ensuite des analyses à des niveaux moins détaillés sans perte initiale d’information. Il faudra toutefois préciser le niveau et la pondération de chaque comparaison. L’absence de données manquantes sur une variable ne prouve pas l’exactitude de toutes ses valeurs. Nous n’avons pas encore produit de résultat scientifique sur l’évolution des statistiques entre générations.

# 03 — Construire la base pour l’analyse

**Suite du notebook 02 :** nous appliquons maintenant les choix de préparation, sans modifier les sources. Une ligne représente **une forme PokéAPI**. Toutes les formes restent présentes, même lorsqu’elles partagent leurs statistiques. Les talents et conditions d’évolution sont exclus.

Les fonctions se trouvent dans `src/preparation.py` et les différences de noms dans `config/preparation.json`. Ce notebook se lance indépendamment des précédents, depuis le dossier du projet ou `notebook/`.

La table commune du notebook 01 est maintenant l’entrée de la préparation. Les deux collectes sont rechargées et comparées ; on construit ensuite la base avec chacune pour vérifier que l’enrichissement donne aussi le même résultat. Le choix `source_pokeapi` dans `config/preparation.json` détermine quelle version sera exportée (CSV par défaut).

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebook":
    project_dir = project_dir.parent
assert (project_dir / "src" / "preparation.py").exists(), "Ouvrir depuis le projet ou notebook/."
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import pandas as pd
from src.collect import (collect_sources, load_sources, check_sources,
                         pokeapi_from_csv, collect_pokeapi_api, compare_pokeapi)
from src import preparation as prep

collect_sources()  # Vérifie les empreintes de l’instantané partagé.
data = load_sources()
source_checks = check_sources(data)

pokeapi_csv = pokeapi_from_csv(data)
pokeapi_api = collect_pokeapi_api()  # Réutilise le cache du notebook 01.
comparaison = compare_pokeapi(pokeapi_csv, pokeapi_api)
display(comparaison["resume"])
display(comparaison["ecarts"].head(20))
assert comparaison["identiques"], "Examiner les écarts CSV/API dans le notebook 01 avant la préparation."


## 1. Quelles variables construit-on ?

| Colonnes | Définition et source |
|---|---|
| `form_id`, `pokemon_id`, `species_id` | Identifiants de la forme (clé unique), de la variété et de l’espèce PokéAPI. |
| `nom_fr`, `forme` | Nom français **de l’espèce**, puis identifiant complet de la forme. Le français des noms de formes n’est pas présent dans les sources collectées : aucune traduction n’est inventée. |
| `generation` | Génération du groupe de jeux d’introduction de **la forme**, indiqué par PokéAPI. |
| `pv`, `attaque`, `defense`, `attaque_speciale`, `defense_speciale`, `vitesse` | Six statistiques de base de la variété à laquelle la forme appartient. |
| `bst` | Somme des six statistiques, pour étudier leur niveau global. |
| `type_1`, `type_2` | Types en français : types propres à la forme lorsqu’ils sont renseignés, sinon types de sa variété. Le second type est vide pour un mono-type. |
| `tag` | Catégories positives identifiées : Légendaire, Fabuleux, Paradoxe, Ultra-Chimère. Plusieurs catégories peuvent coexister, séparées par `; `. |
| `tier` | Champ `tier` de l’instantané Showdown, directement renseigné ou obtenu par une référence documentée. |
| `is_default_form`, `is_battle_only`, `is_mega` | Indicateurs PokéAPI conservés pour décrire la population et permettre des analyses ultérieures à périmètre explicite. |

**Dates et valeurs ne racontent pas la même chose :** la génération est une date d’introduction, tandis que les statistiques sont celles de l’instantané actuel. Nous n’avons pas reconstitué leurs valeurs historiques.

**Rapprochement Showdown :** noms normalisés, noms déclarés dans `baseForme` / `cosmeticFormes`, puis correspondances de libellés explicites dans la configuration. Les statistiques ne servent jamais à choisir une fiche. Le numéro d’espèce, les statistiques et les types sont contrôlés ensuite. Les 63 formes de Charmilly gardent leur ligne : Showdown ne distingue pas leurs sucreries. Les sept couleurs de Météno météore gardent aussi leur ligne malgré une représentation Showdown commune.

In [ ]:
showdown = data["showdown_pokedex"]
base_csv, audit_csv = prep.prepare_database(pokeapi_csv, showdown, source="csv")
base_api, audit_api = prep.prepare_database(pokeapi_api, showdown, source="api")
pd.testing.assert_frame_equal(base_csv, base_api)

source = prep.read_preparation_config()["source_pokeapi"]
assert source in ["csv", "api"], "Choisir csv ou api dans config/preparation.json."
if source == "csv":
    pokeapi, base, audit = pokeapi_csv, base_csv, audit_csv
else:
    pokeapi, base, audit = pokeapi_api, base_api, audit_api
print("Bases finales identiques avec les deux collectes. Source retenue :", source)
display(base.head())

## 2. La fusion conserve-t-elle nos observations ?

La fonction vérifie l’unicité et la conservation de **tous les identifiants de forme**, les rattachements et les variables essentielles. L’audit conserve la fiche choisie, la méthode, les sources des enrichissements et les accords/désaccords. Une divergence de statistiques, de types ou de numéro d’espèce est signalée : elle ne corrige pas les valeurs PokéAPI et bloque l’export tant qu’elle n’a pas été examinée. Une forme sans correspondance reste conservée.

Dans `reference_01`, les **1 579 formes** sont conservées ; les **1 518 rapprochements** donnent des statistiques et types concordants. Ce total désigne des lignes PokéAPI comparées, pas autant de fiches Showdown distinctes. Les 61 lignes restantes ne sont pas vérifiées par Showdown : elles ne constituent ni des accords ni des désaccords.

Les types Showdown sont traduits en identifiants de type avec le dictionnaire de `config/preparation.json`, issu des noms anglais de PokéAPI. Cela permet de comparer les types sans modifier les libellés français de la table commune. `source_collecte` indique CSV/API et `source_generation` conserve l’identifiant du groupe de jeux d’introduction.

In [ ]:
display(prep.summarize_database(base, audit))
display(audit["methode_jointure"].value_counts().rename("formes").to_frame())

# Ce tableau resterait visible si une nouvelle source introduisait une divergence.
divergences = audit.loc[
    audit["stats_concordantes"].eq(False) | audit["types_concordants"].eq(False),
    ["forme", "showdown_id", "stats_divergentes", "types_pokeapi", "types_showdown"]
]
display(divergences)

## 3. Comprendre les absences avant de les traiter

**PokéAPI.** Aucun nom français d’espèce, génération, premier type ou statistique ne manque ici. Les **777 seconds types vides** correspondent aux mono-types : ils restent vides. `arceus-unknown` porte la valeur littérale `???`, distincte d’une donnée manquante ; cette entrée reste dans l’inventaire et devra être isolée explicitement pour une analyse des types jouables.

**Showdown.** Les **61 tiers absents** concernent des correspondances non établies par nos règles :

- 43 formes de Papilord, Lépidonille et Pérégrain, dont les distinctions ne sont pas explicitées par les alias utilisés ;
- 6 formes mâles/femelles de Viskuse, Moyade et Némélios ;
- 8 modes de déplacement de Koraidon et Miraidon ;
- 2 variantes de Zygarde et 1 de Rocabot, non rapprochées ici ;
- `arceus-unknown`.

**Traitement retenu :** conserver ces lignes et laisser leur tier vide. Cela ne prouve pas que les 61 formes sont absentes de Showdown : certaines y sont décrites à un niveau moins détaillé, et d’autres demandent encore une justification de correspondance. On pourra enrichir la configuration plus tard, sans modifier la population de la base. Aucun tier `Illegal` n’est créé pour combler une absence.

Parmi les tiers disponibles, **1 372 sont explicites** dans la fiche choisie et **146 proviennent de références**. Les références de combat ont priorité sur `baseSpecies`, avec les cas Mimikyu/Totem prévus par le [moteur Showdown](https://github.com/smogon/pokemon-showdown/blob/master/sim/dex-species.ts). La règle et la fiche d’origine figurent dans l’audit. Il s’agit d’une lecture documentée du JSON sauvegardé, pas d’une exécution complète du simulateur.

In [ ]:
sans_tier = base.loc[base["tier"].isna()]
display(sans_tier.groupby("nom_fr", sort=True).agg(
    formes=("form_id", "size"), exemple=("forme", "first")
))
display(audit["regle_tier"].value_counts().rename("formes").to_frame())

# Toutes les lignes restent consultables, sans allonger l’affichage du notebook.
formes_a_revoir = audit.loc[audit["showdown_id"].isna()]

## 4. Que signifie le tag ?

Les indicateurs `is_legendary` et `is_mythical` viennent de PokéAPI. Pour les Paradoxes, on combine les preuves positives des tags Showdown et de la catégorie anglaise `Paradox Pokémon` de PokéAPI. Cette dernière couvre **22 espèces**, dont Koraidon, Miraidon et les quatre espèces récentes non étiquetées Paradox dans ce JSON Showdown. Le tag Ultra-Chimère vient des **11 espèces** explicitement étiquetées `Ultra Beast` par Showdown.

Ces catégories sont attribuées **au niveau de l’espèce**, puis transmises à ses formes, même sans correspondance de forme Showdown. Les provenances sont dans `source_tag`. Les autres tags Showdown, par exemple `True Past`, ne sont pas des catégories biologiques retenues pour la comparaison ; ceux de la fiche rapprochée restent consultables dans l’audit.

**`Non étiqueté` signifie qu’aucune de ces catégories n’a été identifiée avec ces critères, pas que la source garantit un Pokémon « ordinaire ».** Une absence de tag ne devient donc pas une fausse certitude. Koraidon et Miraidon cumulent Légendaire et Paradoxe : les catégories ne forment pas une partition. Les effectifs d’espèces évitent de confondre couverture des catégories et multiplication des formes.

In [ ]:
display(base.groupby("tag").agg(
    formes=("form_id", "size"), especes=("species_id", "nunique")
))

# Cas illustrant les choix de représentation, sans les programmer un par un.
exemples = ["charizard", "charizard-gmax", "arceus-fairy", "arceus-unknown",
            "silvally-fairy", "minior-red-meteor", "minior-red", "koraidon-apex-build"]
display(base.loc[base["forme"].isin(exemples),
    ["form_id", "nom_fr", "forme", "generation", "type_1", "type_2", "bst", "tag", "tier"]])

## 5. Export et limites pour l’analyse

Deux CSV régénérables sont écrits dans `data/processed/reference_01/` : `pokemon_forms.csv` pour l’analyse et `audit_preparation.csv` pour retrouver chaque choix grâce à `form_id`. Un troisième fichier, `provenance.json`, conserve la source retenue, les instantanés et les empreintes des sources de référence, du code, de la configuration et des deux CSV exportés. Ce dossier est ignoré par Git ; chacun peut le reconstruire avec le même instantané et la même configuration. Aucun fichier brut n’est modifié.

La base est exhaustive **par rapport aux formes de l’instantané PokéAPI**, sans garantir que la source inventorie parfaitement toute la licence. Les formes cosmétiques, de combat, Gigamax et historiques restent distinctes. Par conséquent, une espèce riche en formes pèse davantage dans une moyenne par ligne : on devra comparer les résultats par forme avec des résultats au niveau variété ou espèce. Les formes d’une espèce ne sont pas des observations indépendantes.

Le tier dépend du contexte compétitif et de la date du JSON ; ce n’est ni un score numérique de puissance ni le tier historique à l’introduction de la forme. Une valeur `Illegal` présente dans la source reste une valeur renseignée. Toute analyse du tier devra annoncer le format étudié et le périmètre retenu, ainsi que les formes sans tier. Les talents, attaques et conditions d’évolution ne sont pas analysés : le BST seul n’explique pas la performance en combat.

In [ ]:
prep.validate_database(base, audit, pokeapi)
output_dir = prep.export_database(base, audit, pokeapi)
print("Base, audit et provenance enregistrés dans :", output_dir)